# DoorDash Delivery ETA — Etapa 06: Baseline e Primeiros Modelos

Este notebook implementa e avalia os primeiros modelos para prever `delivery_duration_minutes` usando a divisão temporal com embargo aprovada:
- **Treino:** `138.572` registros
- **Validação:** `28.808` registros
- **Teste:** `30.041` registros (mantido **completamente isolado**)

> **Regras:** O pré-processamento é ajustado somente no treino. As features confirmadas são o conjunto principal; variáveis operacionais e estimativas de deslocamento são testadas separadamente em experimentos de ablação.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import prepare_datasets
from src.feature_engineering import build_features
from src.temporal_validation import temporal_split_with_embargo
from src.model_training import (
    audit_temporal_split,
    train_and_evaluate_models,
    evaluate_by_duration_band,
    run_ablation_experiments,
    CONFIRMED_MODEL_FEATURES,
    TARGET_COL,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "historical_data.csv"
_, _, df_model, _ = prepare_datasets(RAW_PATH)
df_feat = build_features(df_model)
split = temporal_split_with_embargo(df_feat)

audit = audit_temporal_split(split.train, split.val, split.test, split.val_start, split.test_start)
print("Auditoria pré-treino aprovada:", audit)
print(f"Features confirmadas utilizadas ({len(CONFIRMED_MODEL_FEATURES)}):", CONFIRMED_MODEL_FEATURES)

## 1. Baseline e Modelos Iniciais (Features Confirmadas)

In [ ]:
results_df, preds, fitted = train_and_evaluate_models(split.train, split.val)
display(results_df)

## 2. Análise de Erro por Faixa de Duração Real (Melhor Modelo Provisório)

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
y_val = split.val[TARGET_COL]
y_pred_best = preds[best_model_name]

print(f"Melhor modelo provisório (menor MAE): {best_model_name}")
band_df = evaluate_by_duration_band(y_val, y_pred_best)
display(band_df)

## 3. Gráficos Diagnósticos na Validação

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Real vs Previsto (<= 180 min)
mask_180 = y_val.values <= 180
axes[0, 0].scatter(y_val.values[mask_180], y_pred_best[mask_180], alpha=0.12, s=8, color="#2b5c8f")
axes[0, 0].plot([0, 180], [0, 180], "r--", linewidth=1.5)
axes[0, 0].set_title(f"Reais vs Previstos — {best_model_name} (<= 180 min)")
axes[0, 0].set_xlabel("Duração Real (min)")
axes[0, 0].set_ylabel("Duração Prevista (min)")

# 2. Resíduos
residuals = y_val.values - y_pred_best
sns.histplot(residuals[(residuals >= -60) & (residuals <= 120)], bins=70, kde=True, ax=axes[0, 1], color="#4c72b0")
axes[0, 1].axvline(0, color="red", linestyle="--")
axes[0, 1].set_title("Distribuição dos Resíduos (y_real - y_prev)")
axes[0, 1].set_xlabel("Resíduo (min)")

# 3. Erro por faixa
band_df["MAE"].plot(kind="bar", ax=axes[1, 0], color="#dd8452", label="MAE")
band_df["MedAE"].plot(kind="line", marker="o", ax=axes[1, 0], color="#2b5c8f", linewidth=2, label="MedAE")
axes[1, 0].set_title("Erro Absoluto por Faixa de Duração Real")
axes[1, 0].set_ylabel("Erro (min)")
axes[1, 0].tick_params(axis="x", rotation=20)
axes[1, 0].legend()

# 4. Comparação MAE e RMSE
x = np.arange(len(results_df))
w = 0.35
axes[1, 1].bar(x - w/2, results_df["MAE"], width=w, label="MAE", color="#2b5c8f")
axes[1, 1].bar(x + w/2, results_df["RMSE"], width=w, label="RMSE", color="#c44e52")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(results_df["Model"], rotation=15)
axes[1, 1].set_title("Comparação MAE vs RMSE (Validação)")
axes[1, 1].set_ylabel("Erro (min)")
axes[1, 1].legend()

fig.tight_layout()
plt.show()

## 4. Experimentos de Ablação (EXPERIMENTAIS — Disponibilidade Incerta)

In [ ]:
ablation_df = run_ablation_experiments(split.train, split.val)
display(ablation_df)